# Correlation Power Analysis (Brier et al. 2004)

In [1]:
%load_ext autoreload
%autoreload 2

import os
import random

import lascar
import numpy as np
import plotly.graph_objects as pgo
from cwtoolbox import CaptureDevice

## Exercise 1

In [2]:
capture_device = CaptureDevice.create("CWLITEXMEGA")
capture_device.compile(file=os.path.abspath("../lecture_3/sbox_lookup.c"))
capture_device.flash()

c:\work\securecoding_ws2526\.venv\Lib\site-packages\chipwhisperer\capture\trace\TraceWhisperer.py:31: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources # type: ignore


XMEGA Programming flash...
XMEGA Reading flash...
Verified flash OK, 2553 bytes


In [3]:
data = capture_device.capture(
    number_of_traces=1000,
    input=lambda _: [random.randint(0, 255)] + 15 * [0],
)

100%|██████████| 1000/1000 [00:21<00:00, 47.28it/s]


In [4]:
groups = {i: [] for i in range(9)}
for d in data:
    groups[lascar.hamming(d["input"][0])].append(d["trace"])

for i, g in groups.items():
    groups[i] = np.mean(g, axis=0)

In [5]:
fig = pgo.Figure()
for i, d in groups.items():
    fig.add_trace(pgo.Scatter(y=d, name=f"hw: {i}"))
fig.show()

In [6]:
fig = pgo.Figure()
fig.add_trace(
    pgo.Scatter(x=list(groups.keys()), y=[trace[70] for trace in groups.values()])
)
fig.show()

## Exercise 3

In [7]:
def pearson(x, y):
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    return sum((x - x_mean) * (y - y_mean)) / np.sqrt(
        sum((x - x_mean) ** 2) * sum((y - y_mean) ** 2)
    )

In [8]:
def aes_sbox_cpa(
    traces,
    key_byte_index=0,
    trace_point=0,
):
    pearsons = [
        (
            abs(
                pearson(
                    traces["trace"][:, trace_point],
                    [
                        lascar.hamming(lascar.tools.aes.sbox[j ^ guess])
                        for j in traces["input"][:, key_byte_index]
                    ],
                )
            ),
            guess,
        )
        for guess in range(256)
    ]
    return list(sorted(pearsons, reverse=True))

In [9]:
aes_sbox_cpa(data, trace_point=145)

[(np.float64(0.818041222922326), 1),
 (np.float64(0.21697681375569514), 238),
 (np.float64(0.21245603176011338), 62),
 (np.float64(0.20969932377968709), 15),
 (np.float64(0.20146264380001605), 76),
 (np.float64(0.20045344256108177), 203),
 (np.float64(0.19905418058810134), 23),
 (np.float64(0.19864929983421595), 218),
 (np.float64(0.19825418735866715), 163),
 (np.float64(0.19666896734869854), 244),
 (np.float64(0.19381390329346035), 10),
 (np.float64(0.19192469410237747), 209),
 (np.float64(0.1752785296019206), 229),
 (np.float64(0.1706366340030261), 52),
 (np.float64(0.17009845011862246), 103),
 (np.float64(0.1673373749451586), 82),
 (np.float64(0.16280704340095398), 40),
 (np.float64(0.16159284423879658), 235),
 (np.float64(0.16044032391405552), 166),
 (np.float64(0.15969196767727734), 168),
 (np.float64(0.15882295819196543), 157),
 (np.float64(0.15745221062652567), 250),
 (np.float64(0.15729617655346653), 178),
 (np.float64(0.1567745129910939), 63),
 (np.float64(0.15639803735430466)

In [10]:
# Identify proper trace point

correlation = [
    pearson(
        data["trace"][:, trace_point],
        [lascar.hamming(lascar.tools.aes.sbox[j ^ 0x01]) for j in data["input"][:, 0]],
    )
    for trace_point in range(data["trace"].shape[0])
]

fig = pgo.Figure()
fig.add_trace(pgo.Scatter(y=correlation))
fig.show()

## Exercise 4

In [11]:
import tqdm


def aes_sbox_cpa_2(
    traces,
    key_byte_index=0,
):
    pearsons = []
    for guess in tqdm.tqdm(range(256)):
        pearsons.append(
            np.nanmax(
                [
                    np.abs(
                        pearson(
                            traces["trace"][:, trace_point],
                            [
                                lascar.hamming(lascar.tools.aes.sbox[j ^ guess])
                                for j in traces["input"][:, 0]
                            ],
                        )
                    )
                    for trace_point in range(traces["trace"].shape[1])
                ]
            )
        )
    return list(sorted(pearsons, reverse=True))

In [12]:
aes_sbox_cpa_2(data)

100%|██████████| 256/256 [07:16<00:00,  1.71s/it]


[np.float64(0.8794321817086769),
 np.float64(0.2690557733161803),
 np.float64(0.25621588885642715),
 np.float64(0.25010693146033025),
 np.float64(0.24425006691619783),
 np.float64(0.24216680117664566),
 np.float64(0.23906419582068636),
 np.float64(0.23846769384209968),
 np.float64(0.2378740444000861),
 np.float64(0.2361925426447566),
 np.float64(0.2340351642192547),
 np.float64(0.2337141275318895),
 np.float64(0.2311405077867189),
 np.float64(0.22962983388019387),
 np.float64(0.22806897641924423),
 np.float64(0.22333041116256264),
 np.float64(0.2218777469117418),
 np.float64(0.21698563796289697),
 np.float64(0.2137864846975278),
 np.float64(0.21245603176011338),
 np.float64(0.21226107909973715),
 np.float64(0.21198901381785984),
 np.float64(0.21178934347187522),
 np.float64(0.21116048756586145),
 np.float64(0.21098449624203638),
 np.float64(0.21071727967548368),
 np.float64(0.21024709909843292),
 np.float64(0.20966455744201135),
 np.float64(0.20878293598722508),
 np.float64(0.207747580

## Exercise 5

In [13]:
def selection_function(value, guess):
    return lascar.hamming(lascar.tools.aes.sbox[value["input"][0] ^ guess])


trace = lascar.TraceBatchContainer(data["trace"][:30], data[:30])
engine = lascar.CpaEngine(
    name="engine",
    selection_function=selection_function,
    guess_range=range(256),
)

session = lascar.Session(
    trace,
    engine=engine,
    output_method=lascar.TableOutputMethod(engine),
)
session.run(batch_size="auto")

2026-01-09 09:56:11,179 - lascar.session - INFO - Session Session: 30 traces, 3 engines, batch_size=1228621, leakage_shape=(1376,)
INFO:lascar.session:Session Session: 30 traces, 3 engines, batch_size=1228621, leakage_shape=(1376,)
Session |  0%||0 trc/30 | (3 engines, batch_size=1228621, leakage_shape=(1376,)) |ETA:  --:--:--
2026-01-09 09:56:12,421 - lascar.output.output_method - INFO - engine
INFO:lascar.output.output_method:engine
Session |100%||30 trc/30 | (3 engines, batch_size=1228621, leakage_shape=(1376,)) |ETA:  00:00:00
Session |100%||30 trc/30 | (3 engines, batch_size=1228621, leakage_shape=(1376,)) |Time:  0:00:01


              engine
    rank 1         1
               0.874
    rank 2       141
               0.699
    rank 3        47
                0.69
    rank 4       235
               0.688
    rank 5        74
               0.686
    rank 6       110
               0.679
    rank 7       190
               0.676
    rank 8        95
               0.673
    rank 9        75
                0.67
   rank 10        18
               0.669

